# AI-Based Handwritten Digit Recognition System
### B.Tech Major Project - Artificial Intelligence & Data Science
**Student Name:** Nafize Ali  
**Domain:** Deep Learning & Computer Vision  
**Framework:** TensorFlow & Keras  

---
## Notebook Overview
1. **Environment Setup & Imports**
2. **Dataset Loading & Exploratory Data Analysis (MNIST)**
3. **Data Preprocessing & Normalization**
4. **Convolutional Neural Network (CNN) Architecture**
5. **Baseline Comparison (MLP vs CNN)**
6. **Model Compilation & Training**
7. **Performance Evaluation & Visualizations**
8. **10x10 Confusion Matrix**
9. **Misclassification & Error Analysis**
10. **Model Export for Web Deployment**

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow Version: {tf.__version__}")

## 2. Load MNIST Dataset
The MNIST dataset contains 70,000 $28 \times 28$ grayscale images of handwritten numerals (0-9).

In [ ]:
(x_train_raw, y_train), (x_test_raw, y_test) = tf.keras.datasets.mnist.load_data()

print(f"Training set shape: {x_train_raw.shape}, labels: {y_train.shape}")
print(f"Testing set shape:  {x_test_raw.shape}, labels: {y_test.shape}")
print(f"Pixel value range:  [{x_train_raw.min()}, {x_train_raw.max()}]")

### Visualize Sample MNIST Images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train_raw[i], cmap='gray')
    ax.set_title(f"Label: {y_train[i]}", fontsize=12, fontweight='bold')
    ax.axis('off')
plt.suptitle("Sample MNIST Training Digits", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing & Normalization
1. **Min-Max Normalization:** Scale pixel intensities from $[0, 255]$ to $[0.0, 1.0]$.
2. **Tensor Reshape:** Reshape to $(N, 28, 28, 1)$ to provide the channel dimension for 2D convolutions.

In [ ]:
x_train = x_train_raw.astype("float32") / 255.0
x_test = x_test_raw.astype("float32") / 255.0

x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

print(f"Processed x_train shape: {x_train.shape}")
print(f"Processed x_test shape:  {x_test.shape}")

## 4. Deep CNN Architecture Design

In [ ]:
cnn_model = models.Sequential([
    layers.Input(shape=(28, 28, 1), name="input_layer"),
    
    # Conv Block 1
    layers.Conv2D(32, kernel_size=(3, 3), padding="same", activation="relu", name="conv1_1"),
    layers.BatchNormalization(name="bn1_1"),
    layers.Conv2D(32, kernel_size=(3, 3), activation="relu", name="conv1_2"),
    layers.BatchNormalization(name="bn1_2"),
    layers.MaxPooling2D(pool_size=(2, 2), name="pool1"),
    layers.Dropout(0.25, name="dropout1"),
    
    # Conv Block 2
    layers.Conv2D(64, kernel_size=(3, 3), padding="same", activation="relu", name="conv2_1"),
    layers.BatchNormalization(name="bn2_1"),
    layers.Conv2D(64, kernel_size=(3, 3), activation="relu", name="conv2_2"),
    layers.BatchNormalization(name="bn2_2"),
    layers.MaxPooling2D(pool_size=(2, 2), name="pool2"),
    layers.Dropout(0.25, name="dropout2"),
    
    # Classifier Head
    layers.Flatten(name="flatten"),
    layers.Dense(128, activation="relu", name="dense1"),
    layers.BatchNormalization(name="bn_dense"),
    layers.Dropout(0.40, name="dropout3"),
    layers.Dense(10, activation="softmax", name="output_probabilities")
], name="Handwritten_Digit_CNN")

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_model.summary()

## 5. Model Training with Callbacks

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),
    ModelCheckpoint(filepath="../model/digit_cnn.keras", monitor="val_accuracy", save_best_only=True, verbose=1)
]

history = cnn_model.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

## 6. Training & Validation Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(history.history['accuracy'], 'o-', label='Train Accuracy')
ax1.plot(history.history['val_accuracy'], 's--', label='Validation Accuracy')
ax1.set_title('Accuracy Progression', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.6)

# Loss
ax2.plot(history.history['loss'], 'o-', color='red', label='Train Loss')
ax2.plot(history.history['val_loss'], 's--', color='purple', label='Validation Loss')
ax2.set_title('Cross-Entropy Loss Progression', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## 7. Model Evaluation on 10,000 Unseen Test Images

In [ ]:
test_loss, test_acc = cnn_model.evaluate(x_test, y_test, verbose=0)
print(f"Final Test Accuracy: {test_acc * 100:.2f}%")
print(f"Final Test Loss:     {test_loss:.4f}")

y_probs = cnn_model.predict(x_test, batch_size=256, verbose=0)
y_pred = np.argmax(y_probs, axis=1)

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=[f'Digit {i}' for i in range(10)]))

## 8. Confusion Matrix Heatmap

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(9, 7.5), dpi=150)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=range(10), yticklabels=range(10))
plt.title('10x10 Confusion Matrix (MNIST Test Set)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('Actual True Label', fontsize=12)
plt.tight_layout()
plt.show()

## 9. Error Analysis & Misclassifications

In [ ]:
misclassified_idx = np.where(y_pred != y_test)[0]
print(f"Total misclassifications: {len(misclassified_idx)} / 10,000 samples ({len(misclassified_idx)/10000*100:.2f}% error rate)")

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    idx = misclassified_idx[i]
    ax.imshow(x_test[idx].squeeze(), cmap='gray_r')
    ax.set_title(f"True: {y_test[idx]} | Pred: {y_pred[idx]}\nConf: {y_probs[idx][y_pred[idx]]*100:.1f}%", color='red', fontsize=10, fontweight='bold')
    ax.axis('off')
plt.suptitle("Sample Misclassified MNIST Test Digits", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()